In [4]:
import pandas as pd

# =========================
# LOAD DATASETS
# =========================

# Freight fact table
fact_freight = pd.read_csv("/Users/hamid/Desktop/SupplyChain_Project/data/facts/fact_freight_market.csv")

# Disruption era dimension
dim_era = pd.read_csv("/Users/hamid/Desktop/SupplyChain_Project/data/curated/dim_disruption_era.csv")

# =========================
# PREPARE DATE/YEAR
# =========================

# Convert date_sk to string first
fact_freight["date_sk"] = fact_freight["date_sk"].astype(str)

# Extract year from YYYYMMDD format
fact_freight["year"] = fact_freight["date_sk"].str[:4].astype(int)

# =========================
# FUNCTION TO ASSIGN era_sk
# =========================

def assign_era_sk(year, era_df):
    match = era_df[
        (era_df["start_year"] <= year) &
        (era_df["end_year"] >= year)
    ]

    if not match.empty:
        return match.iloc[0]["era_sk"]

    return None

# Apply lookup logic
fact_freight["era_sk"] = fact_freight["year"].apply(
    lambda x: assign_era_sk(x, dim_era)
)

# =========================
# OPTIONAL VALIDATION
# =========================

# Check for unmatched years
missing_eras = fact_freight["era_sk"].isnull().sum()

print(f"Missing era assignments: {missing_eras}")

# =========================
# FINAL CLEANUP
# =========================

# Drop helper year column if not needed
fact_freight.drop(columns=["year"], inplace=True)

# =========================
# SAVE UPDATED FACT TABLE
# =========================

fact_freight.to_csv(
    "/Users/hamid/Desktop/SupplyChain_Project/data/facts/fact_freight_market_enriched.csv",
    index=False
)

print("fact_freight_market_enriched.csv created successfully.")

Missing era assignments: 0
fact_freight_market_enriched.csv created successfully.


In [6]:
import pandas as pd

# =========================================
# LOAD DATASETS
# =========================================

# Port logistics fact table
fact_port = pd.read_csv("/Users/hamid/Desktop/SupplyChain_Project/data/facts/fact_port_logestics.csv")

# Disruption era dimension
dim_era = pd.read_csv("/Users/hamid/Desktop/SupplyChain_Project/data/curated/dim_disruption_era.csv")

# =========================================
# PREPARE DATE/YEAR
# =========================================

# Ensure date_key is string
fact_port["date_key"] = fact_port["date_key"].astype(str)

# Extract year from YYYYMMDD format
fact_port["year"] = fact_port["date_key"].str[:4].astype(int)

# =========================================
# ERA LOOKUP FUNCTION
# =========================================

def assign_era_sk(year, era_df):
    """
    Assigns era_sk based on year range lookup
    """

    matched_era = era_df[
        (era_df["start_year"] <= year) &
        (era_df["end_year"] >= year)
    ]

    if not matched_era.empty:
        return matched_era.iloc[0]["era_sk"]

    return None

# =========================================
# APPLY ERA ENRICHMENT
# =========================================

fact_port["era_sk"] = fact_port["year"].apply(
    lambda x: assign_era_sk(x, dim_era)
)

# =========================================
# VALIDATION
# =========================================

# Check missing era assignments
missing_era_count = fact_port["era_sk"].isnull().sum()

print(f"Missing era assignments: {missing_era_count}")

# =========================================
# CLEANUP
# =========================================

# Remove helper column
fact_port.drop(columns=["year"], inplace=True)

# =========================================
# OPTIONAL COLUMN REORDERING
# =========================================

fact_port = fact_port[
    [
        "port_fact_sk",
        "date_key",
        "geo_sk",
        "era_sk",
        "avg_wait_days",
        "vessels_anchor",
        "utilization_pct",
        "congestion_level",
        "load_timestamp"
    ]
]

# =========================================
# EXPORT ENRICHED FACT TABLE
# =========================================

fact_port.to_csv(
    "/Users/hamid/Desktop/SupplyChain_Project/data/facts/fact_port_logistics_enriched.csv",
    index=False
)

print("fact_port_logistics_enriched.csv created successfully.")

Missing era assignments: 0
fact_port_logistics_enriched.csv created successfully.
